# 6. Score binder confidence (binding score)

Every returned complex carries a *binding score* — the folding model's own confidence in the antibody-antigen interface it built, read from its confidence head. Used as a screen, it ranks a panel of candidate binders against a target so the most promising rise to the top.

The model exposes three signals — `binding_score_1`, `binding_score_2`, `binding_score_3` — that read the interface in complementary ways. No single one is best on every dataset: pick the signal that best separates your known binders, or, before you have data to choose, average the three by rank within each antigen (what this notebook demos).

This notebook needs the input CSV (to group predictions by antigen) and the confidence JSONs from notebook 2. No ground truth is required.

## How it works

For each prediction we read the three binding-score signals from its `confidence.summary`, group predictions by antigen (from the input CSV), percentile-rank each signal within the antigen group, and average the three ranks into a consensus score. Higher is a stronger predicted binder.

## Outputs

A per-prediction CSV with the antigen group, the three raw binding-score signals, their within-antigen percentile ranks, and the consensus (mean-rank) binding score — sorted best-first within each antigen.

In [ ]:
import csv
import json
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.append("src")
from az_adapter import csv_to_jobs

# Input CSV that drove inference in notebook 2 (used here only to group predictions by antigen).
INPUT_CSV = Path("examples/structure_determination_input.csv")

# Most-recent invoke run under outputs/; override by setting an absolute path.
_invoke_runs = sorted(Path("./outputs").glob("invoke_*"))
CONFIDENCE_DIR = _invoke_runs[-1] / "confidence" if _invoke_runs else Path("./outputs/invoke_<timestamp>/confidence")
OUTPUT_CSV = Path("./outputs/binding_scores.csv")

# The three binding-score signals surfaced by the model's confidence head.
SIGNALS = ["binding_score_1", "binding_score_2", "binding_score_3"]

## Load the three signals

We read each prediction's `confidence.summary` and pull out the three signals, tagging every prediction with the antigen it was folded against. Predictions that share an antigen sequence are grouped under one short `antigen_id`.

In [ ]:
# Map each prediction's input_name to its antigen. We read the raw `antigen_seq` field from
# the CSV (all antigen chains, slash-separated) so multi-chain antigens group correctly -- the
# job's sequences[0] would only capture the first chain. Predictions sharing an antigen get one
# antigen_id.
csv_rows = list(csv.DictReader(INPUT_CSV.open(newline="")))
name_to_antigen = {
    job["name"]: row["antigen_seq"]
    for row, job in zip(csv_rows, csv_to_jobs(INPUT_CSV))
}
unique_antigens = dict.fromkeys(name_to_antigen.values())            # de-duplicated, in input order
antigen_ids = {seq: f"antigen_{i + 1}" for i, seq in enumerate(unique_antigens)}

# Strip the "batch_NNN_nM_aaK_" prefix that notebook 2 prepends to each saved file.
BATCH_PREFIX_RE = re.compile(r"^batch_\d+_n\d+_aa\d+_")


def input_name_from_path(path: Path) -> str:
    return BATCH_PREFIX_RE.sub("", path.stem)


confidence_paths = sorted(CONFIDENCE_DIR.glob("*.json"))
print(f"Found {len(confidence_paths)} confidence JSONs in {CONFIDENCE_DIR}")
if not confidence_paths:
    raise ValueError(f"No confidence JSONs found in {CONFIDENCE_DIR}. Run notebook 2 first to produce predictions.")

rows = []
for path in confidence_paths:
    data = json.loads(path.read_text())
    summary = data.get("summary", data)              # handler may nest under 'summary' or return it flat
    input_name = input_name_from_path(path)
    antigen_seq = name_to_antigen.get(input_name)
    row = {"input_name": input_name, "antigen_id": antigen_ids.get(antigen_seq, "antigen_unknown")}
    row.update({s: summary.get(s) for s in SIGNALS})
    rows.append(row)

df = pd.DataFrame(rows)

# Every binding-score signal must be present for every prediction; a missing
# signal would make the consensus rank average over fewer signals and silently
# bias incomplete predictions upward, so we refuse to run rather than mislead.
for _signal in SIGNALS:
    _missing = df.loc[df[_signal].isna(), "input_name"].tolist()
    if _missing:
        _hint = " (requires an antibody binder)" if _signal == "binding_score_2" else ""
        raise ValueError(
            f"{_signal} missing for {len(_missing)} prediction(s): {_missing}{_hint}. "
            f"All of {SIGNALS} are required for the consensus rank; re-run inference or "
            "remove these predictions before scoring."
        )
# Every prediction must map to an antigen from INPUT_CSV; an unmatched prediction would
# be lumped into one 'antigen_unknown' group and ranked against unrelated binders, so we
# refuse to run rather than emit misleading rankings.
unmatched = df.loc[df["antigen_id"] == "antigen_unknown", "input_name"].tolist()
if unmatched:
    raise ValueError(
        f"{len(unmatched)} prediction(s) did not match any INPUT_CSV row by name: {unmatched}. "
        "Use the same CSV that drove inference so every prediction maps to an antigen."
    )
df.head()

## Rank-average within each antigen

For each antigen we percentile-rank the candidate binders by each signal (1.0 = best in that group), then average the three ranks. Per-antigen ranking is what makes the three comparable: a signal can sit at a different absolute level from one antigen to the next, but *within* a target the ordering is what matters for a screen, so a rank-average there is well defined even when the raw scales differ.

In [ ]:
# Percentile-rank each signal within its antigen (1.0 = best), then average the three ranks.
scored = df.copy()
for s in SIGNALS:
    scored[f"{s}_rank"] = scored.groupby("antigen_id")[s].rank(pct=True)
scored["consensus_rank"] = scored[[f"{s}_rank" for s in SIGNALS]].mean(axis=1)
scored = scored.sort_values(["antigen_id", "consensus_rank"], ascending=[True, False])

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
scored.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(scored)} rows to {OUTPUT_CSV}\n")

# Highest-ranked candidate per antigen.
for antigen_id, group in scored.groupby("antigen_id"):
    top = group.iloc[0]
    print(f"{antigen_id}: top binder = {top['input_name']} (consensus rank {top['consensus_rank']:.2f})")

scored.head(10)